In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()  # Load variables from .env
# LangSmith Configuration
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"  # Fixed: Was overwriting API_KEY
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")
# Other Keys
groq_api_key = os.getenv("GROQ_API_KEY")
os.environ["HUGGING_FACE"] = os.getenv("HUGGING_FACE")

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
emb = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


C:\Users\admin\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from langchain_groq import ChatGroq
llm = ChatGroq(
    temperature=0,
    model_name="llama-3.1-8b-instant"  # or other valid models below
)

In [5]:
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001E38001D2D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001E380782B10>, model_name='llama-3.1-8b-instant', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********'))

In [6]:
import sys
print(sys.executable)


c:\Program Files\Python311\python.exe


In [7]:
import langchain
print(langchain.__version__)


1.2.7


In [8]:

from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.prompts import ChatMessagePromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
# UPDATED IMPORTS: Use langchain_classic for chains
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [9]:
# Remove the strict filtering to allow the loader to capture the content
loader = WebBaseLoader(
    web_paths=("https://en.wikipedia.org/wiki/Avengers:_Doomsday",),
)
doc = loader.load()
doc[0].page_content

'\n\n\n\nAvengers: Doomsday - Wikipedia\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nJump to content\n\n\n\n\n\n\n\nMain menu\n\n\n\n\n\nMain menu\nmove to sidebar\nhide\n\n\n\n\t\tNavigation\n\t\n\n\nMain pageContentsCurrent eventsRandom articleAbout WikipediaContact us\n\n\n\n\n\n\t\tContribute\n\t\n\n\nHelpLearn to editCommunity portalRecent changesUpload fileSpecial pages\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nSearch\n\n\n\n\n\n\n\n\n\n\n\nSearch\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nAppearance\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nDonate\n\nCreate account\n\nLog in\n\n\n\n\n\n\n\n\nPersonal tools\n\n\n\n\n\nDonate Create account Log in\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nContents\nmove to sidebar\nhide\n\n\n\n\n(Top)\n\n\n\n\n\n1\nPremise\n\n\n\n\n\n\n\n\n2\nCast\n\n\n\n\n\n\n\n\n3\nProduction\n\n\n\n\n\n\n\n\n4\nMusic\n\n\n\n\n\n\n\n\n5\nMarketing\n\n\n\n\n\n\n\n\n6\nRelease\n\n\n\n\n\n\n\n\n7\nSequel\n\n\n\n\n\n\n\n\n8\nReferences\n\n\n\n\n\n

In [10]:
text_split=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
split=text_split.split_documents(doc)
vector_store=Chroma.from_documents(split,emb)
ret=vector_store.as_retriever()
ret

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001E381F0CBD0>, search_kwargs={})

In [11]:
# Import the correct class
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "you are an assistant for question and answering tasks."
    "use the following pieces of retrieved context to answer the questions."
    "{context}" 
)

# Use ChatPromptTemplate instead of ChatMessagePromptTemplate
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)




In [12]:
qus=create_stuff_documents_chain(llm,prompt)
rag=create_retrieval_chain(ret,qus)

In [13]:
#rag.invoke({"input":"What is meant by food variety?"})
res=rag.invoke({"input":"director of avengers doomsday?"})

In [14]:
res['answer']

'The directors of Avengers: Doomsday are Anthony and Joe Russo.'

In [15]:
res2=rag.invoke({"input":"What is food?"})

In [16]:
res2['answer']

'Food is any substance that is ingested by an organism to provide nutrition, energy, and promote growth and maintenance of the body. It can be in the form of solid, liquid, or gas, and can be derived from plants, animals, or other sources.\n\nFood provides the body with the necessary nutrients, such as carbohydrates, proteins, fats, vitamins, and minerals, to function properly. These nutrients are broken down and absorbed by the body, where they are used to:\n\n1. Provide energy for physical activity and daily functions\n2. Build and repair tissues, such as muscles, bones, and skin\n3. Support growth and development, particularly in children and adolescents\n4. Maintain proper bodily functions, such as digestion, circulation, and immune function\n\nFood can be categorized into several main groups, including:\n\n1. Carbohydrates (e.g., bread, pasta, fruits, and vegetables)\n2. Proteins (e.g., meat, poultry, fish, eggs, and legumes)\n3. Fats (e.g., oils, nuts, and seeds)\n4. Vitamins and